In [ ]:
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

#### 전처리 대상 데이터 가져오기

##### 민원사무처리기준표 데이터를 기준으로 서류사전 구축

In [ ]:
assis24_value_cnt = pd.read_excel('../P3_Result/01.CSV/03_02_assis24_value_cnt9.xlsx',header = 0) #엑셀데이터 형태상 header지정
compl_value_cnt = pd.read_excel('../P3_Result/01.CSV/03_02_compl_value_cnt9.xlsx',header = 0) #엑셀데이터 형태상 header지정
dictionary_9 =  pd.read_excel('../P1_Data/02.Pretreatment/03_01_dictionary_9.xlsx',index_col=0)
doc_list = pd.read_csv('../P1_Data/02.Pretreatment/무인민원서류목록.csv', encoding = 'cp949')
assis_add = pd.read_csv('../P1_Data/01.External/text_dictionary/assis_add.csv',encoding = 'cp949')


# 전처리 전 컬럼명 통일 수작업 수행
doc_list = pd.concat([doc_list[['민원사무분류명']], assis_add[['민원사무분류명']]], axis=0).reset_index(drop = True)
doc_list.loc[doc_list['민원사무분류명'] == '여권발급기록증명서(국/영)', '민원사무분류명'] = '여권발급기록증명서'
doc_list.loc[doc_list['민원사무분류명'] == '여권실효확인서(국/영)', '민원사무분류명'] = '여권실효확인서'
doc_list.loc[doc_list['민원사무분류명'] == '검정고시 성적증명서(국문, 영문)', '민원사무분류명'] = '검정고시 성적증명서'
doc_list.loc[doc_list['민원사무분류명'] == '검정고시 합격증명서(국문, 영문)', '민원사무분류명'] = '검정고시 합격증명서'
doc_list

In [ ]:
dictionary_9

In [ ]:
assis_12 = assis24_value_cnt[(assis24_value_cnt['빈도수'] == 1) | (assis24_value_cnt['빈도수'] == 2)] # 보조금 24 데이터 1차 빈도수 결과에서 빈도수가 낮은 1,2만 추출
assis_12.reset_index()
assis_12

In [ ]:
compl_12 = compl_value_cnt[(compl_value_cnt['빈도수'] == 1) | (compl_value_cnt['빈도수'] == 2)] # 민원사무처리기준표 데이터 1차 빈도수 결과에서 빈도수가 낮은 1,2만 추출
compl_12

In [ ]:
df_12 = pd.concat([assis_12, compl_12])  #빈도수가 낮은 경우들을 합친 후 해당 서류 삭제
df_12 = df_12.drop_duplicates(['extracted_word'])
df_12

In [ ]:
# df_12['extracted_word']에서 정확히 일치하는 문자열만을 포함하는 리스트 생성
exact_matches = [re.escape(word) for word in df_12['extracted_word']]
# exact_matches에 포함된 문자열만 삭제하는 방법으로 value 데이터프레임에서 필터링
result = dictionary_9[~dictionary_9['구비서류_okt적용후_한글추출'].isin(exact_matches)]
result = result.drop_duplicates(['구비서류_okt적용후_한글추출'])
result

In [ ]:
doc_list = doc_list.rename(columns = {'민원사무분류명':'구비서류_okt적용후_한글추출'}) # df2의 컬럼명을 A로 변경
result_dictionary = pd.concat([result[['구비서류_okt적용후_한글추출']], doc_list[['구비서류_okt적용후_한글추출']]], axis=0).reset_index(drop = True) # df1과 df2를 세로 방향으로 합치기
result_dictionary = result_dictionary.drop_duplicates(['구비서류_okt적용후_한글추출'])
result_dictionary

#### 최종사전 생성

In [ ]:
result_dictionary.to_excel('../P1_Data/02.Pretreatment/03_03_dictionary.xlsx')

In [ ]:
result_dictionary